In [ ]:
import pandas as pd
import numpy as np
import os

Import Aquaculture chemicals data

In [ ]:
data = pd.read_csv('./Data/MAIN_chemical_structure_data.tsv', sep='\t', dtype=str)
data = data.replace(np.nan, '', regex=True)
print(data.shape)
data.head()

In [ ]:
data['pubchemid'] = data['pubchemid'].replace('CID:', '', regex=True)
data['casrn'] = data['casrn'].replace('CAS:', '', regex=True)
print(data.shape)
data.head()

In [ ]:
aquachems = set(list(data['casrn'])) - {''}
len(aquachems)

Import Species

In [ ]:
aquachemspecies_df = pd.read_csv('./external_data/aquaculture_relevant_species.txt', dtype=str, header=None)
print(aquachemspecies_df.shape)

test_species = set(list(aquachemspecies_df[0])) - {''}
print(len(test_species))

In [ ]:
data = data[['casrn', 'pubchemid', 'id']].drop_duplicates()
print(data.shape)
data.head()

### CTD Chemical-Genes

In [ ]:
ctd_genes = pd.read_csv('../../../CTD/Data/CTD_chem_gene_ixns.tsv', sep='\t', dtype=str, comment='#')
print(ctd_genes.shape)
ctd_genes = ctd_genes.replace(np.nan, '')
print(ctd_genes.shape)
ctd_genes.head()

In [ ]:
len(set(ctd_genes['Organism']))

In [ ]:
test_species.intersection(set(ctd_genes['Organism']))
print(len(test_species.intersection(set(ctd_genes['Organism']))))

In [ ]:
ctd_genes_aqua = ctd_genes[ctd_genes['Organism'].isin(test_species)]
print(ctd_genes_aqua.shape)
ctd_genes_aqua.head()

In [ ]:
aquagenes = set(ctd_genes_aqua['GeneID']) - {''}
print(len(aquagenes))

In [ ]:
aquagenesym = set(ctd_genes_aqua['GeneSymbol']) - {''}
print(len(aquagenesym))

In [ ]:
ctd_genes_aqua = ctd_genes_aqua[ctd_genes_aqua['CasRN'].isin(aquachems)]
print(ctd_genes_aqua.shape)
ctd_genes_aqua.head()

In [ ]:
print(len(set(ctd_genes_aqua['Organism'])))
print(len(set(ctd_genes_aqua['CasRN'])))

In [ ]:
ctd_genes_aqua_triplets = ctd_genes_aqua[['CasRN', 'GeneID', 'GeneForms', 'InteractionActions', 'Organism', 'PubMedIDs']].drop_duplicates()
print(ctd_genes_aqua_triplets.shape)
ctd_genes_aqua_triplets.head()

In [ ]:
ctd_genes_aqua_triplets['Endp'] = ctd_genes_aqua_triplets['CasRN'] + '|' + ctd_genes_aqua_triplets['GeneID']
print(ctd_genes_aqua_triplets.shape)
ctd_genes_aqua_triplets.head()

For Main SQL Tables

In [ ]:
ctd_genes_aqua['InteractionActions'].value_counts()

We will remove:

secretion \
oxidation \
abundance \
cotreatment \
transport \
reaction \
response 

In [ ]:
to_remove_actions1 = {
'secretion',
'oxidation',
'abundance',
'cotreatment',
'transport',
'reaction',
'response' 
}


def actionRemove(row):
    rw = []
    for k in to_remove_actions1:
        if k in row:
            rw.append('Y')
        else:
            rw.append('N')

    if 'Y' in rw:
        return 'Y'
    else:
        return 'N'

In [ ]:
ctd_genes_aqua['Relevant_actions'] = ctd_genes_aqua['InteractionActions'].apply(actionRemove)
print(ctd_genes_aqua.shape)
ctd_genes_aqua.head()

In [ ]:
ctd_genes_aqua_triplets_modified = ctd_genes_aqua[ctd_genes_aqua['Relevant_actions']=='N']
ctd_genes_aqua_triplets_modified['InteractionActions'] = ctd_genes_aqua_triplets_modified['InteractionActions'].apply(lambda x: str(x).split('|'))
ctd_genes_aqua_triplets_modified = ctd_genes_aqua_triplets_modified.explode(['InteractionActions'])
print(ctd_genes_aqua_triplets_modified.shape)
ctd_genes_aqua_triplets_modified.head()

In [ ]:
set(ctd_genes_aqua_triplets_modified['GeneForms'])

In [ ]:
ctd_genes_aqua_triplets_modified = ctd_genes_aqua_triplets_modified[ctd_genes_aqua_triplets_modified['GeneForms']!='']
print(ctd_genes_aqua_triplets_modified.shape)
ctd_genes_aqua_triplets_modified.head()

In [ ]:
ctd_genes_aqua_triplets_modified = ctd_genes_aqua_triplets_modified.merge(data, left_on='CasRN', right_on='casrn', how='inner')
print(ctd_genes_aqua_triplets_modified.shape)
ctd_genes_aqua_triplets_modified.head()

In [ ]:
len(set(ctd_genes_aqua_triplets_modified['id']))

In [ ]:
ctd_genes_aqua_triplets_modified = ctd_genes_aqua_triplets_modified[['id', 'pubchemid','casrn','GeneForms','GeneID','GeneSymbol','InteractionActions','OrganismID', 'Organism', 'PubMedIDs']]
print(ctd_genes_aqua_triplets_modified.shape)
ctd_genes_aqua_triplets_modified.head()

In [ ]:
ctd_genes_aqua_triplets_modified = ctd_genes_aqua_triplets_modified.groupby(by=['id', 'pubchemid', 'casrn', 'GeneForms', 'GeneID', 'GeneSymbol',
       'InteractionActions', 'OrganismID', 'Organism']).agg(lambda x: ','.join(list(set(x.to_list())-{''}))).reset_index()
ctd_genes_aqua_triplets_modified['InteractionActions'] = ctd_genes_aqua_triplets_modified['InteractionActions'].str.replace('^', ' ').str.capitalize()
print(ctd_genes_aqua_triplets_modified.shape)
ctd_genes_aqua_triplets_modified.head()

### CTD Chemical-Phenotypes

In [ ]:
ctd_pheno = pd.read_csv('../../../CTD/Data/CTD_pheno_term_ixns.tsv', sep='\t', dtype=str, comment='#')
print(ctd_pheno.shape)
ctd_pheno = ctd_pheno.replace(np.nan, '')
print(ctd_pheno.shape)
ctd_pheno.head()

In [ ]:
len(set(ctd_pheno['organism']))

In [ ]:
test_species.intersection(set(ctd_pheno['organism']))
print(len(test_species.intersection(set(ctd_pheno['organism']))))

In [ ]:
ctd_pheno_aqua = ctd_pheno[(ctd_pheno['organism'].isin(test_species))&(ctd_pheno['casrn'].isin(aquachems))]
print(ctd_pheno_aqua.shape)
ctd_pheno_aqua.head()

In [ ]:
print(len(set(ctd_pheno_aqua['organism'])))
print(len(set(ctd_pheno_aqua['casrn'])))

In [ ]:
ctd_pheno_aqua_triplets = ctd_pheno_aqua[['casrn', 'phenotypeid', 'interactionactions', 'organism', 'pubmedids']].drop_duplicates()
print(ctd_pheno_aqua_triplets.shape)
ctd_pheno_aqua_triplets.head()

In [ ]:
ctd_pheno_aqua_triplets['Endp'] = ctd_pheno_aqua_triplets['casrn'] + '|' + ctd_pheno_aqua_triplets['phenotypeid']
print(ctd_pheno_aqua_triplets.shape)
ctd_pheno_aqua_triplets.head()

For Main SQL Tables

In [ ]:
set(ctd_pheno_aqua['interactionactions'])

In [ ]:
to_remove_actions2 = {
'secretion',
'oxidation',
'abundance',
'cotreatment',
'transport',
'reaction',
'response',
'metabolic',
'uptake' 
}


def actionRemove2(row):
    rw = []
    for k in to_remove_actions2:
        if k in row:
            rw.append('Y')
        else:
            rw.append('N')

    if 'Y' in rw:
        return 'Y'
    else:
        return 'N'

In [ ]:
ctd_pheno_aqua['Relevant_actions'] = ctd_pheno_aqua['interactionactions'].apply(actionRemove2)
print(ctd_pheno_aqua.shape)
ctd_pheno_aqua.head()

In [ ]:
ctd_pheno_aqua_triplets_modified = ctd_pheno_aqua[ctd_pheno_aqua['Relevant_actions']=='N']
ctd_pheno_aqua_triplets_modified['interactionactions'] = ctd_pheno_aqua_triplets_modified['interactionactions'].apply(lambda x: str(x).split('|'))
ctd_pheno_aqua_triplets_modified = ctd_pheno_aqua_triplets_modified.explode(['interactionactions'])
print(ctd_pheno_aqua_triplets_modified.shape)
ctd_pheno_aqua_triplets_modified.head()

In [ ]:
ctd_pheno_aqua_triplets_modified = ctd_pheno_aqua_triplets_modified.merge(data, left_on='casrn', right_on='casrn', how='inner')
print(ctd_pheno_aqua_triplets_modified.shape)
ctd_pheno_aqua_triplets_modified.head()

In [ ]:
len(set(ctd_pheno_aqua_triplets_modified['ID']))

In [ ]:
ctd_pheno_aqua_triplets_modified[ctd_pheno_aqua_triplets_modified['ID']=='AquachemID12']

In [ ]:
ctd_pheno_aqua_triplets_modified = ctd_pheno_aqua_triplets_modified[['ID', 'pubchemid', 'casrn', 'phenotypeid', 'phenotypename', 'interactionactions', 'organismid', 'organism', 'pubmedids']].drop_duplicates()
print(ctd_pheno_aqua_triplets_modified.shape)
ctd_pheno_aqua_triplets_modified.head()

In [ ]:
ctd_pheno_aqua_triplets_modified.columns

In [ ]:
ctd_pheno_aqua_triplets_modified = ctd_pheno_aqua_triplets_modified.groupby(by=['ID', 'pubchemid', 'casrn', 'phenotypeid', 'phenotypename',
       'interactionactions', 'organismid', 'organism']).agg(lambda x: ','.join(list(set(x.to_list())-{''}))).reset_index()
ctd_pheno_aqua_triplets_modified['interactionactions'] = ctd_pheno_aqua_triplets_modified['interactionactions'].str.replace('^', ' ').str.capitalize()
print(ctd_pheno_aqua_triplets_modified.shape)
ctd_pheno_aqua_triplets_modified.head()

### CTD Gene-Disease

In [ ]:
ctd_gene_dis = pd.read_csv('../Data/CTD_curated_genes_diseases.tsv', sep='\t', dtype=str, comment='#')
print(ctd_gene_dis.shape)
ctd_gene_dis = ctd_gene_dis.replace(np.nan, '')
print(ctd_gene_dis.shape)
ctd_gene_dis.head()

In [ ]:
set(ctd_gene_dis['DirectEvidence'])

In [ ]:
ctd_gene_dis = ctd_gene_dis[ctd_gene_dis['DirectEvidence'].isin({'marker/mechanism', 'marker/mechanism|therapeutic'})]
print(ctd_gene_dis.shape)
ctd_gene_dis.head()

In [ ]:
ctd_gene_dis = ctd_gene_dis[ctd_gene_dis['GeneID'].isin(aquagenes)]
print(ctd_gene_dis.shape)
ctd_gene_dis.head()

### CTD Chemical-Disease

In [ ]:
ctd_chem_dis = pd.read_csv('../Data/CTD_curated_chemicals_diseases.tsv', sep='\t', dtype=str, comment='#')
print(ctd_chem_dis.shape)
ctd_chem_dis = ctd_chem_dis.replace(np.nan, '')
print(ctd_chem_dis.shape)
ctd_chem_dis.head()

In [ ]:
ctd_chem_dis_aqua = ctd_chem_dis[ctd_chem_dis['CasRN'].isin(aquachems)]
print(ctd_chem_dis_aqua.shape)
ctd_chem_dis_aqua.head()

In [ ]:
ctd_chem_dis_aqua = ctd_chem_dis_aqua[ctd_chem_dis_aqua['DirectEvidence'].isin({'marker/mechanism', 'marker/mechanism|therapeutic'})]
print(ctd_chem_dis_aqua.shape)
ctd_chem_dis_aqua.head()

In [ ]:
len(set(ctd_chem_dis_aqua['CasRN']) - {''})

In [ ]:
# chunking the data 
def process_gene2go_file(filename, column_name, target_value, cnksize):
    filtered_chunks = [
        chunk[chunk[column_name].isin(target_value)] 
        for chunk in pd.read_csv(filename, chunksize=cnksize, sep='\t',dtype=str)
    ]
    return pd.concat(filtered_chunks) if filtered_chunks else pd.DataFrame()

In [ ]:
def Get_combinations(lis):
    # consider all the tetramers from the list
    tetramers = []
    chemical = lis[0][0]
    gene = lis[0][1]
    for k in lis[1]: # looping through phenotypes
        for l in lis[2]: # looping through diseases
            phenotype = k
            disease = l
            tetramers.append([chemical, gene, phenotype, disease])
    return tetramers  

## Gene2GO Gene-Phenotype

In [ ]:
gene_GO_file = '../Data/gene2go'

In [ ]:
gene_GO_CAS = process_gene2go_file(gene_GO_file, 'GeneID', list(set(ctd_genes_aqua['GeneID'])-{''}), 10000) 
gene_GO_CAS.replace(np.nan, '', regex=True, inplace=True)
print(gene_GO_CAS.shape)
gene_GO_CAS.head()

In [ ]:
# these are the experimental evidence codes for GO terms listed in https://geneontology.org/docs/guide-go-evidence-codes/
# We are only considering the experimental evidence for the GO terms from NCBI to get the gene-GO association

GO_evidence = ['EXP','IDA','IPI','IMP','IGI','IEP','HTP','HDA','HMP','HGI','HEP']

gene_GO_CAS_exp_evidence = pd.DataFrame(gene_GO_CAS[gene_GO_CAS['Evidence'].isin(GO_evidence)].reset_index(drop=True))
print(gene_GO_CAS_exp_evidence.shape)
gene_GO_CAS_exp_evidence.head()

In [ ]:
print(len(list(gene_GO_CAS_exp_evidence['GeneID'].unique())))
print(len(list(ctd_genes_aqua['GeneID'].unique())))

In [ ]:
gene_mapper = dict(set(zip(ctd_genes_aqua['GeneID'], ctd_genes_aqua['GeneSymbol'])))
print(len(gene_mapper))

In [ ]:
pheno_mapper = dict(set(zip(ctd_pheno_aqua['phenotypeid'], ctd_pheno_aqua['phenotypename'])))
print(len(pheno_mapper))

In [ ]:
dis_mapper = dict(set(zip(ctd_chem_dis['DiseaseID'], ctd_chem_dis['DiseaseName'])))
print(len(dis_mapper))

## Construct tetramers

In [ ]:
ctd_genes_aqua.columns

In [ ]:
# Finding chemical-gene (CG) tuple
# this step considers chemical-gene data
chem_gene_tuple = tuple(set(tuple(zip(ctd_genes_aqua['CasRN'],ctd_genes_aqua['GeneID']))))
print(len(chem_gene_tuple))

In [ ]:
ctd_pheno_aqua.columns

In [ ]:
# Finding chemical gene phenotype map
# this step considers chemical-phenotype data and gene-phenotype data

chem_gene_phenotype = {}
for ele in chem_gene_tuple:
    chem_gene_phenotype[ele] = list(set(ctd_pheno_aqua.loc[ctd_pheno_aqua['casrn'] == ele[0],'phenotypeid']).intersection(set(gene_GO_CAS_exp_evidence.loc[gene_GO_CAS_exp_evidence['GeneID'] ==ele[1],'GO_ID'])))
print(len(chem_gene_phenotype))  

In [ ]:
ctd_chem_dis_aqua.columns

In [ ]:
# Finding chemical gene disease map
# this step considers chemical-disease data, gene-disease data

chem_gene_disease = {}
for ele in chem_gene_tuple:
    chem_gene_disease[ele] = list(set(ctd_chem_dis_aqua.loc[ctd_chem_dis_aqua['CasRN'] == ele[0],'DiseaseID']).intersection(set(ctd_gene_dis.loc[ctd_gene_dis['GeneID'] ==ele[1],'DiseaseID'])))
print(len(chem_gene_disease))

In [ ]:
tetramers = []
# for (chemical, gene) as keys, check if it has atleast some phenotypes and diseases 
for ele in chem_gene_disease.keys():
    if len(chem_gene_phenotype[ele]) >=1 and len(chem_gene_disease[ele]) >=1:
        tetramers.append(Get_combinations([ele,chem_gene_phenotype[ele], chem_gene_disease[ele]]))

print(len(tetramers))

In [ ]:
tetramers_list = []
chemical_list = []
gene_list = []
phenotype_list = []
disease_list = []

for k in tetramers:
    for tetramer in k:
        tetramers_list.append(tetramer)
        
        chemical_list.append(tetramer[0])
        gene_list.append(tetramer[1])
        phenotype_list.append(tetramer[2])
        disease_list.append(tetramer[3])

In [ ]:
print('Number of total tetramers:', len(tetramers_list))
print('Number of total chemicals:', len(set(chemical_list)))
print('Number of total genes:', len(set(gene_list)))
print('Number of total phenotypes:', len(set(phenotype_list)))
print('Number of total diseases:', len(set(disease_list)))

In [ ]:
with open('../Output/Disease_list_CAS.tsv','w') as f1:
    for ele in set(disease_list):
        f1.write(ele+'\t'+ctd_gene_dis.loc[ctd_gene_dis['DiseaseID']==ele,'DiseaseName'].iloc[0]+'\n')

with open('../Output/Tetramer_list_CAS.tsv','w') as f2:
    f2.write('Chemical'+'\t'+'Gene'+'\t'+'Phenotype'+'\t'+'Disease'+'\n')

    for ele in tetramers_list:
        f2.write(ele[0]+'\t'+ele[1]+'\t'+ele[2]+'\t'+ele[3]+'\n')

In [ ]:
tetramers_df = pd.DataFrame(tetramers_list, columns=['CAS', 'GeneID', 'PhenotypeID', 'DiseaseID'])
print(tetramers_df.shape)
tetramers_df.head()

In [ ]:
tetramers_df.to_csv('../Output/Tetramers_for_manual_check.tsv', sep='\t', index=False, encoding='utf-8')

## Chem-Gene table

In [ ]:
ctd_genes_aqua_triplets_modified.head()

In [ ]:
print(len(set(ctd_genes_aqua_triplets_modified['ID'])))
print(len(set(ctd_genes_aqua_triplets_modified['GeneID'])))
print(len(set(ctd_genes_aqua_triplets_modified['Organism'])))

In [ ]:
ctd_genes_aqua_triplets_modified.to_csv('../Output/Final_chem_gene.tsv', sep='\t', index=False, encoding='utf-8')

## Chem-Pheno table

In [ ]:
ctd_pheno_aqua_triplets_modified.head()

In [ ]:
ctd_pheno_aqua_triplets_modified.columns

In [ ]:
ctd_pheno_aqua_triplets_modified.columns = ['ID', 'pubchemid', 'casrn', 'PhenotypeID', 'PhenotypeName',
       'InteractionActions', 'organismID', 'Organism', 'PubMedIds']

In [ ]:
print(len(set(ctd_pheno_aqua_triplets_modified['ID'])))
print(len(set(ctd_pheno_aqua_triplets_modified['PhenotypeID'])))
print(len(set(ctd_pheno_aqua_triplets_modified['Organism'])))

In [ ]:
ctd_pheno_aqua_triplets_modified.to_csv('../Output/Final_chem_pheno.tsv', sep='\t', index=False, encoding='utf-8')